# AI Job Agent — PHASE 6: Cross-Encoder Re-ranking

**Goal:** take the leading field-by-field matches from Phase 5 and re-rank them with a multilingual Cross-Encoder that reads the candidate and each job **together**. This captures semantic relationships that exact field matching cannot express.

**Single-candidate scope.** This notebook implements **PATH B (live candidate)**, consistent with Phases 4 and 5. It does not generate user-facing explanations — that remains Phase 7.

```text
matched_jobs.parquet (Phase 5)       candidate_profile.json (Phase 2)
               │                                  │
               └──── privacy-safe text pairs ─────┘
                                  │
                   Multilingual Cross-Encoder
                                  │
                  reranked_jobs.parquet
                  final_top_jobs.json
                        (→ Phase 7)
```

Phase 6 does not invent a fixed weighted sum across Phase 3/5/6. The Cross-Encoder score is the primary re-ranking signal; Phase 5 scores and Phase 3 similarity are retained as independent deterministic tie-breakers and audit features.

---
## 1. Setup

In [ ]:
import gc
import hashlib
import json
import math
import shutil
import subprocess
import sys
import time
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.max_columns", 60)

WORK = Path("work")
OUT = Path("outputs")
for d in (WORK, OUT):
    d.mkdir(parents=True, exist_ok=True)

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M")
print("RUN_ID:", RUN_ID)

---
## 2. Runtime dependencies

The reranker and its external custom model code are pinned to immutable revisions. This model is not compatible with Transformers 5 because its rotary-position buffer can be left uninitialized, producing huge invalid position indices during inference. The setup therefore enforces the tested `transformers==4.53.3` runtime before importing Transformers.

In [ ]:
from importlib.metadata import PackageNotFoundError, version as package_version

# Tested compatibility boundary for Alibaba-NLP/new-impl. Transformers 5 can
# leave the model's non-persistent RoPE position buffer uninitialized.
REQUIRED_PACKAGES = [
    "transformers==4.53.3",
    "accelerate>=1.0,<2",
    "safetensors>=0.4",
]
PINNED_TRANSFORMERS = "4.53.3"

try:
    installed_transformers = package_version("transformers")
except PackageNotFoundError:
    installed_transformers = None

if installed_transformers != PINNED_TRANSFORMERS:
    print(f"Installing tested Transformers runtime (found {installed_transformers!r})...")
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", "--upgrade",
        *REQUIRED_PACKAGES,
    ])

import torch
import transformers
assert transformers.__version__ == PINNED_TRANSFORMERS, (
    f"Expected transformers {PINNED_TRANSFORMERS}, got {transformers.__version__}. "
    "Restart the runtime once, then run all cells again."
)

from transformers import AutoModelForSequenceClassification, AutoTokenizer

print("torch       :", torch.__version__)
print("transformers:", transformers.__version__)
print("CUDA        :", torch.cuda.is_available())

---
## 3. Load Phase 5 output

Upload `matched_jobs.parquet` from Phase 5's `outputs/live_candidate/` folder and `run_metadata.json` from the Phase 5 output root. You may also upload a previous `reranker_cache.jsonl` to reuse identical candidate/job/model scores.

In [ ]:
from google.colab import files as _colab_files

print("Upload matched_jobs.parquet + run_metadata.json from Phase 5")
print("Optional: upload reranker_cache.jsonl from an earlier Phase 6 run.")
uploaded_p5 = _colab_files.upload()

P5_DIR = WORK / "phase5"
P5_DIR.mkdir(exist_ok=True)
for name, content in uploaded_p5.items():
    (P5_DIR / name).write_bytes(content)

p5_file = P5_DIR / "matched_jobs.parquet"
p5_meta_path = P5_DIR / "run_metadata.json"
assert p5_file.exists(), "matched_jobs.parquet not found among uploaded files."
assert p5_meta_path.exists(), "run_metadata.json from Phase 5 is required for provenance checks."
print("Saved Phase 5 inputs.")

In [ ]:
matched_df = pd.read_parquet(p5_file)
phase5_meta = json.loads(p5_meta_path.read_text(encoding="utf-8"))

required_cols = {
    "candidate_id", "job_id", "phase5_rank", "objective_rank_score",
    "match_score", "scoring_confidence", "similarity",
    "is_phase6_candidate", "title_clean", "jd_clean",
}
missing = required_cols - set(matched_df.columns)
assert not missing, f"matched_jobs.parquet is missing expected Phase 5 columns: {missing}"
assert matched_df["job_id"].is_unique, "duplicate job_id in Phase 5 output"
assert matched_df["match_score"].between(0, 100).all(), "invalid match_score"
assert matched_df["objective_rank_score"].between(0, 100).all(), "invalid objective_rank_score"

p5_candidate_id = phase5_meta.get("candidate_id")
file_candidate_ids = set(matched_df["candidate_id"].dropna().astype(str))
assert len(file_candidate_ids) <= 1, "Phase 5 file contains multiple candidate ids"
if file_candidate_ids and p5_candidate_id:
    assert next(iter(file_candidate_ids)) == str(p5_candidate_id), "Phase 5 metadata candidate mismatch"

print(f"Phase 5 jobs: {len(matched_df):,}")
print(f"Phase 5 scoring version: {phase5_meta.get('scoring_version')}")
print(f"Candidate id: {p5_candidate_id}")

---
## 4. Load candidate profile (Phase 2)

Upload `candidate_profile.json`. Phase 6 uses professional content only. Name, email, phone, source filename and other identity fields are excluded and explicitly redacted from the model input.

In [ ]:
print("Upload candidate_profile.json from Phase 2:")
uploaded_profile = _colab_files.upload()

P2_DIR = WORK / "phase2"
P2_DIR.mkdir(exist_ok=True)
for name, content in uploaded_profile.items():
    (P2_DIR / name).write_bytes(content)

profile_path = P2_DIR / "candidate_profile.json"
assert profile_path.exists(), "candidate_profile.json not found."
candidate_profile = json.loads(profile_path.read_text(encoding="utf-8"))

assert candidate_profile.get("candidate_id"), "candidate_profile.json has no candidate_id"
if p5_candidate_id:
    assert str(candidate_profile["candidate_id"]) == str(p5_candidate_id), (
        "candidate_profile.json does not belong to the candidate in Phase 5"
    )
print(f"Loaded candidate {candidate_profile['candidate_id']} "
      f"({candidate_profile.get('most_recent_title')})")

---
## 5. Configuration

The multilingual reranker is pinned to a reviewed commit. Raw model logits are retained as ranking scores; they are **not probabilities** and are never presented as a percent chance of job fit.

In [ ]:
CONFIG = {
    "reranker_model": "Alibaba-NLP/gte-multilingual-reranker-base",
    "model_revision": "8215cf04918ba6f7b6a62bb44238ce2953d8831c",
    "remote_code_revision": "40ced75c3017eb27626c9d4ea981bde21a2662f4",
    "trust_remote_code": True,
    "phase5_pool_size": 50,
    "final_top_n": 20,
    "initial_batch_size": 8,
    # Operational memory bound, not a matching weight. Structured evidence is placed first.
    "max_pair_tokens": 2048,
}

assert CONFIG["model_revision"] != "main", "Pin an immutable model revision"
assert CONFIG["remote_code_revision"] != "main", "Pin the external custom-code revision"
assert CONFIG["final_top_n"] <= CONFIG["phase5_pool_size"]
print(json.dumps(CONFIG, indent=2))

---
## 6. Select the Phase 6 candidate pool

Phase 6 never reintroduces a job removed by Phase 4 or excluded from the Phase 5 leading pool. If the flag is unexpectedly empty, it falls back to the first `phase5_pool_size` rows rather than producing a silent empty result.

In [ ]:
flag = matched_df["is_phase6_candidate"].fillna(False).astype(bool)
if flag.any():
    rerank_pool = matched_df.loc[flag].copy()
else:
    rerank_pool = matched_df.sort_values("phase5_rank").head(CONFIG["phase5_pool_size"]).copy()
    if len(matched_df):
        print("[WARNING] No is_phase6_candidate rows; using the leading Phase 5 rows.")

rerank_pool = rerank_pool.sort_values("phase5_rank").head(CONFIG["phase5_pool_size"]).reset_index(drop=True)
assert rerank_pool["job_id"].is_unique
print(f"Phase 6 pool: {len(rerank_pool):,}/{len(matched_df):,} Phase 5 jobs")
if len(rerank_pool) < 2:
    print("[NOTE] Fewer than two jobs: semantic relevance can be scored, but no comparative re-ranking is possible.")

---
## 7. Build privacy-safe candidate and job texts

Structured fields appear before free text so they survive right-side truncation. PII is excluded by construction and exact identity values are redacted defensively if they appear inside a summary.

In [ ]:
PII_FIELDS = ["full_name", "email", "phone", "source_file"]


def cell_values(value) -> list:
    if value is None:
        return []
    if isinstance(value, (list, tuple, set, np.ndarray, pd.Series)):
        items = list(value)
    else:
        try:
            if pd.isna(value):
                return []
        except (TypeError, ValueError):
            pass
        items = [value]
    return [str(v).strip() for v in items if v is not None and str(v).strip()
            and str(v).strip().lower() not in {"nan", "none", "not specified"}]


def joined(value) -> str:
    return ", ".join(cell_values(value))


def text_section(label: str, value) -> str:
    values = cell_values(value)
    return f"{label}: {', '.join(values)}" if values else ""


def redact_pii(text: str, profile: dict) -> str:
    redacted = text
    for field in PII_FIELDS:
        value = profile.get(field)
        if value and len(str(value).strip()) >= 3:
            redacted = redacted.replace(str(value), "[REDACTED]")
    return redacted


def build_candidate_text(profile: dict) -> str:
    sections = [
        text_section("Current role", profile.get("most_recent_title")),
        text_section("Previous roles", profile.get("past_titles")),
        text_section("Hard skills", profile.get("hard_skills_norm")),
        text_section("Tools and technologies", profile.get("tools_technologies_norm")),
        text_section("Soft skills", profile.get("soft_skills_norm")),
        text_section("Languages", profile.get("languages_norm")),
        text_section("Experience level", profile.get("experience_level")),
        text_section("Total years of experience", profile.get("years_experience_total")),
        text_section("Industries", profile.get("industries")),
        text_section("Education level", profile.get("education_level")),
        text_section("Education fields", profile.get("education_field")),
        text_section("Professional qualifications", profile.get("qualifications")),
        text_section("Professional summary", profile.get("summary")),
    ]
    return redact_pii("\n".join(s for s in sections if s), profile)


def build_job_text(row: pd.Series) -> str:
    years = []
    if pd.notna(row.get("years_experience_min")):
        years.append(f"minimum {row.get('years_experience_min')}")
    if pd.notna(row.get("years_experience_max")):
        years.append(f"maximum {row.get('years_experience_max')}")
    sections = [
        text_section("Job title", row.get("title_clean")),
        text_section("Industry", row.get("industry")),
        text_section("Department", row.get("department")),
        text_section("Required skills", row.get("required_skills_norm")),
        text_section("Preferred skills", row.get("preferred_skills_norm")),
        text_section("Hard skills", row.get("hard_skills_norm")),
        text_section("Tools and technologies", row.get("tools_technologies_norm")),
        text_section("Soft skills", row.get("soft_skills_norm")),
        text_section("Languages", row.get("languages_norm")),
        text_section("Experience level", row.get("experience_level")),
        text_section("Experience years", years),
        text_section("Education level", row.get("education_level")),
        text_section("Education fields", row.get("education_field")),
        text_section("Qualifications", row.get("qualifications")),
        text_section("Responsibilities", row.get("responsibilities")),
        text_section("Full job description", row.get("jd_clean")),
    ]
    return "\n".join(s for s in sections if s)


def sha256_text(text: str) -> str:
    return hashlib.sha256(text.encode("utf-8")).hexdigest()


candidate_text = build_candidate_text(candidate_profile)
job_texts = [build_job_text(row) for _, row in rerank_pool.iterrows()]
candidate_text_hash = sha256_text(candidate_text)

assert candidate_text.strip(), "candidate text is empty"
for field in PII_FIELDS:
    value = candidate_profile.get(field)
    if value and len(str(value).strip()) >= 3:
        assert str(value) not in candidate_text, f"PII field leaked into candidate text: {field}"

rerank_pool["candidate_text_hash"] = candidate_text_hash
rerank_pool["job_text_hash"] = [sha256_text(text) for text in job_texts]
print(f"Candidate text: {len(candidate_text):,} characters")
print(f"Job texts: {len(job_texts):,}")

### 7.1 Text-builder sanity checks

In [ ]:
_test_profile = {
    "full_name": "Jane Example", "email": "jane@example.com", "phone": "123456789",
    "source_file": "jane_cv.pdf", "most_recent_title": "Data Analyst",
    "hard_skills_norm": ["sql"], "summary": "Jane Example uses SQL.",
}
_test_text = build_candidate_text(_test_profile)
assert "Data Analyst" in _test_text and "sql" in _test_text
assert "Jane Example" not in _test_text and "jane@example.com" not in _test_text
assert sha256_text("same") == sha256_text("same")
assert sha256_text("same") != sha256_text("different")
print("Text construction and privacy sanity checks passed.")

---
## 8. Load the pinned multilingual reranker

The model is loaded once. CUDA uses float16; CPU uses float32. If loading fails, the notebook continues with a transparent Phase 5 fallback instead of losing the candidate's results.

In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32
tokenizer = None
reranker_model = None
model_load_error = None

try:
    tokenizer = AutoTokenizer.from_pretrained(
        CONFIG["reranker_model"],
        revision=CONFIG["model_revision"],
        trust_remote_code=CONFIG["trust_remote_code"],
    )
    reranker_model = AutoModelForSequenceClassification.from_pretrained(
        CONFIG["reranker_model"],
        revision=CONFIG["model_revision"],
        code_revision=CONFIG["remote_code_revision"],
        trust_remote_code=CONFIG["trust_remote_code"],
        torch_dtype=DTYPE,
    ).to(DEVICE)
    reranker_model.eval()

    # Fail early on tokenizer/model mismatches and the Transformers-5 RoPE bug.
    smoke_pairs = [["data scientist with Python", "Python data science role"]]
    smoke_inputs = tokenizer(
        smoke_pairs, padding=True, truncation=True, max_length=64, return_tensors="pt"
    )
    max_token_id = int(smoke_inputs["input_ids"].max())
    model_vocab_size = int(reranker_model.config.vocab_size)
    assert 0 <= max_token_id < model_vocab_size, (
        f"Tokenizer/model vocabulary mismatch: token {max_token_id}, vocab {model_vocab_size}"
    )
    smoke_inputs = {k: v.to(DEVICE) for k, v in smoke_inputs.items()}
    with torch.inference_mode():
        smoke_score = reranker_model(
            **smoke_inputs, return_dict=True
        ).logits.view(-1).float().cpu()
    assert smoke_score.numel() == 1 and torch.isfinite(smoke_score).all()

    print(f"Loaded {CONFIG['reranker_model']} on {DEVICE} ({DTYPE})")
    print(f"Model smoke test passed; score={float(smoke_score[0]):.4f}")
except Exception as exc:
    model_load_error = f"{type(exc).__name__}: {exc}"
    reranker_model = None
    tokenizer = None
    print("[WARNING] Reranker could not be loaded. Phase 5 fallback will be used.")
    print(model_load_error)

### 8.1 Token counts and truncation diagnostics

In [ ]:
pair_token_counts = []
input_was_truncated = []

if tokenizer is not None:
    for job_text in job_texts:
        token_ids = tokenizer(
            candidate_text, job_text, add_special_tokens=True, truncation=False
        )["input_ids"]
        count = len(token_ids)
        pair_token_counts.append(count)
        input_was_truncated.append(count > CONFIG["max_pair_tokens"])
else:
    pair_token_counts = [None] * len(job_texts)
    input_was_truncated = [False] * len(job_texts)

rerank_pool["pair_token_count"] = pair_token_counts
rerank_pool["input_was_truncated"] = input_was_truncated
print(f"Pairs requiring truncation: {sum(input_was_truncated)}/{len(input_was_truncated)}")

---
## 9. Score cache

A score is reusable only when candidate text, job text, model id, model weights, custom-code revision, and tested Transformers runtime all match. The cache is exported with the outputs and can be uploaded on a later rerun.

In [ ]:
CACHE_INPUT = P5_DIR / "reranker_cache.jsonl"
CACHE_OUTPUT = OUT / "reranker_cache.jsonl"


def pair_cache_key(candidate_hash: str, job_hash: str) -> str:
    payload = "|".join([
        candidate_hash, job_hash, CONFIG["reranker_model"],
        CONFIG["model_revision"], CONFIG["remote_code_revision"],
        transformers.__version__,
    ])
    return sha256_text(payload)


def load_cache(path: Path) -> dict:
    records = {}
    if not path.exists():
        return records
    for line in path.read_text(encoding="utf-8").splitlines():
        if not line.strip():
            continue
        try:
            record = json.loads(line)
            if "key" in record and "raw_score" in record:
                records[record["key"]] = record
        except json.JSONDecodeError:
            print("[WARNING] Ignoring malformed cache line.")
    return records


score_cache = load_cache(CACHE_INPUT)
rerank_pool["pair_cache_key"] = [
    pair_cache_key(candidate_text_hash, job_hash) for job_hash in rerank_pool["job_text_hash"]
]
print(f"Reusable cache records loaded: {len(score_cache):,}")

---
## 10. Adaptive batched inference

Only cache misses are sent to the model. GPU out-of-memory errors reduce the batch size until one; other errors trigger a documented Phase 5 fallback.

In [ ]:
def score_pairs_once(model, tokenizer, pairs: list, batch_size: int) -> tuple:
    scores = []
    per_pair_ms = []
    for start in range(0, len(pairs), batch_size):
        batch = pairs[start:start + batch_size]
        batch_started = time.perf_counter()
        encoded = tokenizer(
            batch, padding=True, truncation=True,
            max_length=CONFIG["max_pair_tokens"], return_tensors="pt",
        )
        encoded = {k: v.to(DEVICE) for k, v in encoded.items()}
        with torch.inference_mode():
            logits = model(**encoded, return_dict=True).logits.view(-1).float().cpu().numpy()
        elapsed_ms = (time.perf_counter() - batch_started) * 1000.0
        scores.extend(float(v) for v in logits)
        per_pair_ms.extend([elapsed_ms / len(batch)] * len(batch))
    return scores, per_pair_ms


def score_pairs_adaptive(model, tokenizer, pairs: list, initial_batch_size: int) -> tuple:
    if not pairs:
        return [], [], initial_batch_size
    batch_size = max(1, min(initial_batch_size, len(pairs)))
    while True:
        try:
            scores, times = score_pairs_once(model, tokenizer, pairs, batch_size)
            return scores, times, batch_size
        except RuntimeError as exc:
            is_oom = "out of memory" in str(exc).lower()
            if not is_oom or batch_size == 1:
                raise
            batch_size = max(1, batch_size // 2)
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            gc.collect()
            print(f"[WARNING] GPU OOM; retrying with batch_size={batch_size}")

In [ ]:
raw_scores = [None] * len(rerank_pool)
inference_times_ms = [0.0] * len(rerank_pool)
cache_hit = [False] * len(rerank_pool)
rerank_error = model_load_error
effective_batch_size = None

miss_indices, miss_pairs = [], []
for i, (key, job_text) in enumerate(zip(rerank_pool["pair_cache_key"], job_texts)):
    if key in score_cache:
        raw_scores[i] = float(score_cache[key]["raw_score"])
        cache_hit[i] = True
    else:
        miss_indices.append(i)
        miss_pairs.append([candidate_text, job_text])

if miss_pairs and reranker_model is not None:
    try:
        new_scores, new_times, effective_batch_size = score_pairs_adaptive(
            reranker_model, tokenizer, miss_pairs, CONFIG["initial_batch_size"]
        )
        for idx, score, elapsed_ms in zip(miss_indices, new_scores, new_times):
            raw_scores[idx] = score
            inference_times_ms[idx] = elapsed_ms
            key = rerank_pool.iloc[idx]["pair_cache_key"]
            score_cache[key] = {
                "key": key, "raw_score": score,
                "model": CONFIG["reranker_model"],
                "revision": CONFIG["model_revision"],
                "code_revision": CONFIG["remote_code_revision"],
                "transformers_version": transformers.__version__,
                "created_utc": datetime.now(timezone.utc).isoformat(),
            }
    except Exception as exc:
        rerank_error = f"{type(exc).__name__}: {exc}"
        print("[WARNING] Inference failed; preserving Phase 5 order.")
        print(rerank_error)

rerank_pool["reranker_raw_score"] = pd.to_numeric(pd.Series(raw_scores), errors="coerce")
rerank_pool["inference_time_ms"] = inference_times_ms
rerank_pool["cache_hit"] = cache_hit

CACHE_OUTPUT.write_text(
    "\n".join(json.dumps(record, ensure_ascii=False, sort_keys=True)
                 for record in score_cache.values()),
    encoding="utf-8",
)
print(f"Cache hits: {sum(cache_hit)}/{len(cache_hit)}")
print(f"New model scores: {len(miss_pairs) if rerank_error is None else 0}")

---
## 11. Re-rank the pool

A successful model score is the primary ordering signal. Phase 5 objective/absolute scores, confidence, Phase 3 similarity and the old rank break exact ties deterministically. If model scoring is unavailable, Phase 5 order is preserved.

In [ ]:
def finalize_ranking(pool: pd.DataFrame, model_error=None) -> pd.DataFrame:
    out = pool.copy()
    score_available = out["reranker_raw_score"].notna().all() if len(out) else False

    if score_available:
        out = out.sort_values(
            ["reranker_raw_score", "objective_rank_score", "match_score",
             "scoring_confidence", "similarity", "phase5_rank"],
            ascending=[False, False, False, False, False, True],
            kind="mergesort",
        ).reset_index(drop=True)
        if len(out) == 1:
            status = "no_comparison_pool"
        else:
            status = "success"
    else:
        out = out.sort_values("phase5_rank", kind="mergesort").reset_index(drop=True)
        status = "fallback_phase5" if len(out) else "empty_input"

    out.insert(0, "phase6_rank", range(1, len(out) + 1))
    out["rank_change"] = out["phase5_rank"].astype(int) - out["phase6_rank"]
    out["rerank_status"] = status
    out["rerank_error"] = model_error
    out["reranker_model"] = CONFIG["reranker_model"]
    out["reranker_revision"] = CONFIG["model_revision"]
    out["reranker_code_revision"] = CONFIG["remote_code_revision"]
    out["transformers_version"] = transformers.__version__

    if score_available and len(out):
        out["reranker_percentile"] = (
            out["reranker_raw_score"].rank(method="average", pct=True) * 100.0
        ).round(2)
    else:
        out["reranker_percentile"] = np.nan
    return out


reranked_df = finalize_ranking(rerank_pool, rerank_error)
print(f"Re-ranked jobs: {len(reranked_df)}")
print("Status:", reranked_df["rerank_status"].iloc[0] if len(reranked_df) else "empty_input")

### 11.1 Ranking sanity checks

In [ ]:
_test_pool = pd.DataFrame({
    "job_id": ["a", "b", "c"],
    "phase5_rank": [1, 2, 3],
    "reranker_raw_score": [0.1, 0.9, 0.5],
    "objective_rank_score": [90.0, 80.0, 70.0],
    "match_score": [90.0, 80.0, 70.0],
    "scoring_confidence": [0.9, 0.9, 0.9],
    "similarity": [0.8, 0.7, 0.6],
})
_test_ranked = finalize_ranking(_test_pool)
assert _test_ranked["job_id"].tolist() == ["b", "c", "a"]
assert _test_ranked["phase6_rank"].tolist() == [1, 2, 3]
assert _test_ranked.loc[0, "rank_change"] == 1
_test_pool["reranker_raw_score"] = np.nan
_test_fallback = finalize_ranking(_test_pool, "test error")
assert _test_fallback["job_id"].tolist() == ["a", "b", "c"]
assert (_test_fallback["rerank_status"] == "fallback_phase5").all()
print("Ranking and fallback sanity checks passed.")

---
## 12. Review and diagnostics

In [ ]:
display_cols = [
    "phase6_rank", "phase5_rank", "rank_change", "reranker_raw_score",
    "reranker_percentile", "objective_rank_score", "match_score",
    "title_clean", "company_clean", "matched_core_skills",
    "missing_core_skills", "rerank_status",
]
display_cols = [c for c in display_cols if c in reranked_df.columns]
reranked_df[display_cols].head(CONFIG["final_top_n"]).style.format({
    "reranker_raw_score": lambda x: "—" if pd.isna(x) else f"{x:.4f}",
    "reranker_percentile": lambda x: "—" if pd.isna(x) else f"{x:.1f}",
    "objective_rank_score": "{:.2f}",
    "match_score": "{:.2f}",
})

In [ ]:
if len(reranked_df) == 0:
    print("[WARNING] No Phase 5 jobs were available for re-ranking.")
else:
    status = reranked_df["rerank_status"].iloc[0]
    print("Rerank status:", status)
    print(f"Cache hits: {int(reranked_df['cache_hit'].sum())}/{len(reranked_df)}")
    print(f"Truncated inputs: {int(reranked_df['input_was_truncated'].sum())}/{len(reranked_df)}")
    print(f"Total new inference time: {reranked_df['inference_time_ms'].sum():.1f} ms")
    if status == "success":
        print("\nRaw reranker score distribution (not probabilities):")
        print(reranked_df["reranker_raw_score"].describe().round(4).to_string())
        moved = reranked_df.loc[reranked_df["rank_change"] != 0, [
            "title_clean", "phase5_rank", "phase6_rank", "rank_change"
        ]]
        print(f"\nJobs whose rank changed: {len(moved)}/{len(reranked_df)}")
    elif status == "no_comparison_pool":
        print("[NOTE] The model scored one job, but no comparative reorder was possible.")
    else:
        print("[WARNING] Model scoring was unavailable; Phase 5 order was preserved.")

---
## 13. Save outputs

`reranked_jobs.parquet` is the authoritative full output. `final_top_jobs.json` is a compact, privacy-safe Phase 7 payload containing only the final leading jobs and factual matching evidence.

In [ ]:
LIVE_OUT = OUT / "live_candidate"
LIVE_OUT.mkdir(parents=True, exist_ok=True)

reranked_path = LIVE_OUT / "reranked_jobs.parquet"
reranked_df.to_parquet(reranked_path, index=False, compression="snappy")

summary_cols = [
    "phase6_rank", "phase5_rank", "rank_change", "candidate_id", "job_id",
    "title_clean", "company_clean", "reranker_raw_score",
    "reranker_percentile", "objective_rank_score", "match_score",
    "scoring_confidence", "similarity", "matched_core_skills",
    "missing_core_skills", "rerank_status",
]
summary_cols = [c for c in summary_cols if c in reranked_df.columns]
summary_df = reranked_df[summary_cols].copy()
for col in ["matched_core_skills", "missing_core_skills"]:
    if col in summary_df.columns:
        summary_df[col] = summary_df[col].map(lambda values: " | ".join(cell_values(values)))
summary_path = LIVE_OUT / "reranking_summary.csv"
summary_df.to_csv(summary_path, index=False, encoding="utf-8-sig")

top_df = reranked_df.head(CONFIG["final_top_n"])
phase7_jobs = []
for _, row in top_df.iterrows():
    phase7_jobs.append({
        "job_id": row.get("job_id"),
        "phase6_rank": int(row.get("phase6_rank")),
        "phase5_rank": int(row.get("phase5_rank")),
        "title": row.get("title_clean"),
        "company": row.get("company_clean"),
        "url": row.get("url_clean"),
        "location": {"city": row.get("city"), "country": row.get("country")},
        "job_description": row.get("jd_clean"),
        "reranker_raw_score": None if pd.isna(row.get("reranker_raw_score")) else float(row.get("reranker_raw_score")),
        "objective_rank_score": float(row.get("objective_rank_score")),
        "match_score": float(row.get("match_score")),
        "scoring_confidence": float(row.get("scoring_confidence")),
        "matched_core_skills": cell_values(row.get("matched_core_skills")),
        "missing_core_skills": cell_values(row.get("missing_core_skills")),
        "matched_qualifications": cell_values(row.get("matched_qualifications")),
        "missing_qualifications": cell_values(row.get("missing_qualifications")),
        "matched_education_fields": cell_values(row.get("matched_education_fields")),
        "missing_education_fields": cell_values(row.get("missing_education_fields")),
        "rerank_status": row.get("rerank_status"),
    })

phase7_payload = {
    "candidate_id": candidate_profile["candidate_id"],
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "reranker_model": CONFIG["reranker_model"],
    "reranker_revision": CONFIG["model_revision"],
    "reranker_code_revision": CONFIG["remote_code_revision"],
    "transformers_version": transformers.__version__,
    "jobs": phase7_jobs,
}
phase7_path = LIVE_OUT / "final_top_jobs.json"
phase7_path.write_text(json.dumps(phase7_payload, indent=2, ensure_ascii=False, default=str), encoding="utf-8")

RUN_METADATA = {
    "run_id": RUN_ID,
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "candidate_id": candidate_profile["candidate_id"],
    "n_phase5_jobs": int(len(matched_df)),
    "n_reranked_jobs": int(len(reranked_df)),
    "n_final_jobs": int(len(phase7_jobs)),
    "status": reranked_df["rerank_status"].iloc[0] if len(reranked_df) else "empty_input",
    "model_load_error": model_load_error,
    "rerank_error": rerank_error,
    "device": DEVICE,
    "dtype": str(DTYPE),
    "transformers_version": transformers.__version__,
    "effective_batch_size": effective_batch_size,
    "cache_hits": int(sum(cache_hit)),
    "config": CONFIG,
}
(OUT / "run_metadata.json").write_text(
    json.dumps(RUN_METADATA, indent=2, ensure_ascii=False), encoding="utf-8"
)

print(f"Saved {reranked_path} ({len(reranked_df)} rows)")
print(f"Saved {summary_path} ({len(summary_df)} rows)")
print(f"Saved {phase7_path} ({len(phase7_jobs)} jobs)")
print(f"Saved {CACHE_OUTPUT} ({len(score_cache)} cached scores)")

---
## 14. Data dictionary

In [ ]:
DATA_DICTIONARY = '''# Phase 6 outputs — Data Dictionary

## live_candidate/reranked_jobs.parquet

One row per Phase 5 pool job. All Phase 5 columns and factual evidence are
preserved. No job filtered out before Phase 6 can re-enter the result.

| Column | Notes |
|---|---|
| phase6_rank | Final Phase 6 rank, contiguous from 1. |
| reranker_raw_score | Cross-Encoder logit used for ordering. Not a probability. |
| reranker_percentile | Within-pool percentile for display only. Not a match probability. |
| rank_change | phase5_rank - phase6_rank; positive means the job moved up. |
| rerank_status | success / no_comparison_pool / fallback_phase5 / empty_input. |
| pair_token_count | Untruncated tokenizer length for the candidate/job pair. |
| input_was_truncated | True when the pair exceeded CONFIG max_pair_tokens. |
| cache_hit | True when the exact model/candidate/job score was reused. |
| inference_time_ms | Approximate per-pair time for newly inferred batches. |
| reranker_model / reranker_revision | Immutable model provenance. |

## live_candidate/reranking_summary.csv

Human-readable rank comparison and principal matching evidence. The Parquet
file is authoritative.

## live_candidate/final_top_jobs.json

Privacy-safe Phase 7 payload containing at most CONFIG final_top_n jobs, full
job descriptions, scores, and factual matched/missing evidence. It contains
candidate_id but no candidate name, email, phone, source filename, or full
candidate profile.

## reranker_cache.jsonl

One JSON record per exact candidate-text/job-text/model/revision hash. Upload
this file with Phase 5 inputs to reuse scores on an identical rerun.

## Re-ranking contract
- The Cross-Encoder score is primary; Phase 5/3 signals break exact ties.
- No fixed manual fusion weights are used.
- Raw logits are relevance scores, not probabilities.
- A one-job pool is scored but receives no_comparison_pool.
- Any model/load/inference failure preserves Phase 5 ordering.
- Phase 7 explains the final jobs; Phase 6 generates no user-facing prose.
'''

(OUT / "PHASE6_DATA_DICTIONARY.md").write_text(DATA_DICTIONARY, encoding="utf-8")
print("Data dictionary written.")

---
## 15. Final verification

In [ ]:
reloaded = pd.read_parquet(OUT / "live_candidate" / "reranked_jobs.parquet")
final_payload = json.loads((OUT / "live_candidate" / "final_top_jobs.json").read_text(encoding="utf-8"))

assert reloaded["job_id"].is_unique, "duplicate job_id in Phase 6 output"
assert set(reloaded["job_id"]).issubset(set(matched_df["job_id"])), "Phase 6 introduced a new job"
assert len(final_payload["jobs"]) <= CONFIG["final_top_n"]
assert final_payload["candidate_id"] == candidate_profile["candidate_id"]

if len(reloaded):
    assert reloaded["phase6_rank"].tolist() == list(range(1, len(reloaded) + 1))
    assert (reloaded["rank_change"] == reloaded["phase5_rank"] - reloaded["phase6_rank"]).all()
    status = reloaded["rerank_status"].iloc[0]
    if status in {"success", "no_comparison_pool"}:
        assert reloaded["reranker_raw_score"].notna().all()
        assert reloaded["reranker_raw_score"].is_monotonic_decreasing
    elif status == "fallback_phase5":
        assert reloaded["phase5_rank"].is_monotonic_increasing
    print(f"rows       : {len(reloaded)}")
    print(f"top job    : {reloaded.iloc[0].get('title_clean')}")
    print(f"status     : {status}")
else:
    print("rows       : 0")
    print("status     : empty_input")

# Candidate PII must never appear in the Phase 7 JSON payload.
payload_text = json.dumps(final_payload, ensure_ascii=False)
for field in PII_FIELDS:
    value = candidate_profile.get(field)
    if value and len(str(value).strip()) >= 3:
        assert str(value) not in payload_text, f"PII leaked into Phase 7 payload: {field}"

print("STATUS: ready for Phase 7")

---
## 16. Persist outputs

The ZIP contains the full machine-readable result, Phase 7 payload, review CSV, cache, metadata, and data dictionary.

In [ ]:
zip_path = shutil.make_archive("phase6_outputs", "zip", OUT)
print(f"Created {zip_path} ({Path(zip_path).stat().st_size/1e6:.2f} MB)")
_colab_files.download(zip_path)

---
## Next step — PHASE 7

Phase 7 reads `final_top_jobs.json` and produces grounded candidate-facing explanations: why each job fits and what is missing. It must treat the matched/missing evidence as facts, use the job description for context, avoid inventing skills or experience, and keep raw reranker logits internal.

### Method references

- Sentence Transformers Cross-Encoder retrieve-and-rerank pattern: https://www.sbert.net/docs/cross_encoder/usage/usage.html
- Pinned multilingual reranker model card: https://huggingface.co/Alibaba-NLP/gte-multilingual-reranker-base
- mGTE paper/model family: https://arxiv.org/abs/2407.19669